In [ ]:
# 01 - Dataset Consistency Check

This notebook verifies the processed CALCE working files used in the manuscript.

Expected local input files:

- `Processed_Outputv3_1.xlsx`
- `Processed_Outputv3_2.xlsx`
- `Processed_Outputv3_3.xlsx`

The notebook checks sample-number coverage, cycle-index ranges, file-level overlap, observed condition coverage, and consistency with the nominal CALCE 24-condition map.

The large processed Excel input files are not redistributed in this repository. They must be provided locally by the user.

In [ ]:
# ============================================================
# Setup
# ============================================================

# In Colab, uncomment the following line if dependencies are missing:
# !pip -q install openpyxl pandas numpy

import os
import re
import math
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from openpyxl import load_workbook

try:
    from IPython.display import display
except Exception:
    display = print

In [ ]:
# ============================================================
# Input / output configuration
# ============================================================

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

DATA_DIR = "."
OUTPUT_DIR = "outputs/01_dataset_consistency_check"
os.makedirs(OUTPUT_DIR, exist_ok=True)

required_files = [
    "Processed_Outputv3_1.xlsx",
    "Processed_Outputv3_2.xlsx",
    "Processed_Outputv3_3.xlsx"
]

required_paths = [os.path.join(DATA_DIR, f) for f in required_files]

missing_files = [p for p in required_paths if not os.path.exists(p)]

if len(missing_files) == 0:
    print("All required files are already available. Upload step skipped.")
else:
    print("Missing required files:")
    for p in missing_files:
        print(" -", os.path.basename(p))

    if IN_COLAB:
        print("\nPlease upload the missing files.")
        uploaded = files.upload()

        missing_after = [p for p in required_paths if not os.path.exists(p)]

        if missing_after:
            raise FileNotFoundError(
                "The following files are still missing: "
                + ", ".join(os.path.basename(p) for p in missing_after)
            )
    else:
        raise FileNotFoundError(
            "Required input files are missing. Place the three Processed_Outputv3 files "
            "in the notebook working directory."
        )

xlsx_files = sorted(required_paths)

print("\nFiles used:")
for f in xlsx_files:
    print(" -", os.path.basename(f))

In [ ]:
# ------------------------------------------------------------
# 2) Helper functions
# ------------------------------------------------------------

def clean_colname(x):
    x = str(x).strip()
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^A-Za-z0-9_]+", "", x)
    return x.lower()


def normalize_rate(x):
    """
    Discharge C-rate normalization.
    Returns float: 0.7, 1.0, 2.0 etc.
    """
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    s = str(x).strip().upper()
    s = s.replace(" ", "")
    s = s.replace(",", ".")
    s = s.replace("C", "")

    try:
        return float(s)
    except:
        return np.nan


def normalize_temp(x):
    """
    Temperature normalization.
    Returns int/float, usually 10,25,45,60.
    """
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return np.nan

    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(round(float(x)))

    s = str(x).strip().upper()
    s = s.replace("°C", "")
    s = s.replace("ºC", "")
    s = s.replace("C", "")
    s = s.replace(" ", "")
    s = s.replace(",", ".")

    try:
        return int(round(float(s)))
    except:
        return np.nan


def normalize_charge_rate(x):
    """
    Charge cut-off current normalization.
    Returns string: C/5, C/40, etc.
    """
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None

    if isinstance(x, (int, float, np.integer, np.floating)):
        val = float(x)
        if abs(val - 0.2) < 1e-6:
            return "C/5"
        if abs(val - 0.025) < 1e-6:
            return "C/40"
        return str(val)

    s = str(x).strip().upper()
    s = s.replace(" ", "")
    s = s.replace("\\", "/")
    s = s.replace("–", "-")
    s = s.replace("—", "-")

    # Common variants
    if s in ["C/5", "C5", "C-5", "1/5C", "0.2C", "0.20C"]:
        return "C/5"

    if s in ["C/40", "C40", "C-40", "1/40C", "0.025C"]:
        return "C/40"

    # Try to extract denominator from C/number
    m = re.search(r"C/?(\d+)", s)
    if m:
        return f"C/{m.group(1)}"

    return s


def to_int_or_nan(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return np.nan
    try:
        return int(float(x))
    except:
        return np.nan


def condition_key(charge_rate, temp, discharge_rate):
    """
    Canonical condition key:
    (Charge_Rate, Temperature, Discharge_Rate)
    """
    cr = normalize_charge_rate(charge_rate)
    t = normalize_temp(temp)
    r = normalize_rate(discharge_rate)

    if isinstance(r, float) and not np.isnan(r):
        r = round(r, 3)

    return (cr, t, r)


def condition_label(cond):
    cr, temp, rate = cond
    if rate is None or (isinstance(rate, float) and np.isnan(rate)):
        rate_str = "NA"
    else:
        rate_str = f"{rate:g}C"
    return f"{cr} & {temp}°C & {rate_str}"


In [ ]:
# ------------------------------------------------------------
# 3) Expected condition maps
#    A) Full theoretical 192-cell map
#    B) DOCX-reduced map:
#       Condition 23: samples 177-179 only
#       Condition 24: samples 185-187 only
# ------------------------------------------------------------
def build_condition_table(full_192=True):
    temps = [10, 25, 45, 60]
    discharge_rates = [0.7, 1.0, 2.0]
    charge_rates = ["C/5", "C/40"]

    rows = []
    cond_no = 1
    start_sample = 1

    for temp in temps:
        for dr in discharge_rates:
            for cr in charge_rates:
                if full_192:
                    samples = list(range(start_sample, start_sample + 8))
                else:
                    # DOCX version
                    if cond_no == 23:
                        samples = [177, 178, 179]
                    elif cond_no == 24:
                        samples = [185, 186, 187]
                    else:
                        samples = list(range(start_sample, start_sample + 8))

                for s in samples:
                    rows.append({
                        "Condition": cond_no,
                        "sample_number": s,
                        "Expected_Charge_Rate": cr,
                        "Expected_Temperature": temp,
                        "Expected_Discharge_Rate": dr,
                        "Expected_Label": condition_label((cr, temp, dr))
                    })

                cond_no += 1
                start_sample += 8

    return pd.DataFrame(rows)


expected_full = build_condition_table(full_192=True)
expected_docx = build_condition_table(full_192=False)

full_sample_set = set(expected_full["sample_number"])
docx_sample_set = set(expected_docx["sample_number"])

expected_full_map = {
    int(r.sample_number): (
        r.Condition,
        r.Expected_Charge_Rate,
        int(r.Expected_Temperature),
        round(float(r.Expected_Discharge_Rate), 3),
        r.Expected_Label
    )
    for r in expected_full.itertuples(index=False)
}

expected_docx_map = {
    int(r.sample_number): (
        r.Condition,
        r.Expected_Charge_Rate,
        int(r.Expected_Temperature),
        round(float(r.Expected_Discharge_Rate), 3),
        r.Expected_Label
    )
    for r in expected_docx.itertuples(index=False)
}

In [ ]:
# ------------------------------------------------------------
# 4) Column detection
# ------------------------------------------------------------

POSSIBLE_COLUMNS = {
    "sample_number": [
        "sample_number", "samplenumber", "sample", "sample_no",
        "sampleno", "cell", "cell_no", "cellnumber"
    ],
    "rate": [
        "rate", "discharge_rate", "dischargerate",
        "discharge_crate", "c_rate", "crate"
    ],
    "temperature": [
        "temperature", "temp", "ambient_temperature",
        "ambienttemperature"
    ],
    "charge_rate": [
        "charge_rate", "chargerate", "charge_cutoff",
        "charge_cut_off", "charge_cutoff_current",
        "charge_cut_off_current", "cutoff_current",
        "cut_off_current"
    ],
    "cycle_index": [
        "cycle_index", "cycleindex", "cycle", "cycle_no",
        "cycleno", "cycle_number", "cyclenumber"
    ],
    "soc": [
        "soc", "stateofcharge", "state_of_charge"
    ],
    "voltage": [
        "voltagev", "voltage", "voltage_v"
    ],
    "current": [
        "currenta", "current", "current_a"
    ]
}


def detect_columns(header):
    cleaned = [clean_colname(c) for c in header]

    detected = {}
    for canonical, candidates in POSSIBLE_COLUMNS.items():
        detected[canonical] = None
        for idx, cname in enumerate(cleaned):
            if cname in candidates:
                detected[canonical] = idx
                break

    return detected, cleaned

In [ ]:
# ------------------------------------------------------------
# 5) Read Excel files in memory-safe mode
# ------------------------------------------------------------

file_summaries = []
overall_samples = set()
overall_conditions = Counter()
overall_sample_conditions = defaultdict(Counter)
overall_sample_files = defaultdict(set)
overall_sample_cycles = defaultdict(set)
overall_cycle_minmax = defaultdict(lambda: [np.inf, -np.inf])
overall_soc_minmax = defaultdict(lambda: [np.inf, -np.inf])

rows_by_file_condition = Counter()
rows_by_file_sample = Counter()

required_cols = ["sample_number", "rate", "temperature", "charge_rate", "cycle_index"]

for file_name in xlsx_files:
    print(f"\nReading: {file_name}")

    wb = load_workbook(file_name, read_only=True, data_only=True)

    if "Step5_Data" in wb.sheetnames:
        ws = wb["Step5_Data"]
    else:
        ws = wb[wb.sheetnames[0]]

    header = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
    detected, cleaned_header = detect_columns(header)

    missing_required = [c for c in required_cols if detected.get(c) is None]
    if missing_required:
        raise ValueError(
            f"{file_name} dosyasında zorunlu kolonlar bulunamadı: {missing_required}\n"
            f"Bulunan kolonlar: {header}\n"
            f"Temizlenmiş kolonlar: {cleaned_header}"
        )

    idx_sample = detected["sample_number"]
    idx_rate = detected["rate"]
    idx_temp = detected["temperature"]
    idx_charge = detected["charge_rate"]
    idx_cycle = detected["cycle_index"]
    idx_soc = detected.get("soc")

    file_samples = set()
    file_conditions = Counter()
    file_cycle_min = np.inf
    file_cycle_max = -np.inf
    file_rows = 0

    for row in ws.iter_rows(min_row=2, values_only=True):
        file_rows += 1

        sample = to_int_or_nan(row[idx_sample])
        if pd.isna(sample):
            continue

        sample = int(sample)

        rate = normalize_rate(row[idx_rate])
        temp = normalize_temp(row[idx_temp])
        charge = normalize_charge_rate(row[idx_charge])
        cyc = to_int_or_nan(row[idx_cycle])

        cond = (charge, temp, round(rate, 3) if not pd.isna(rate) else np.nan)

        file_samples.add(sample)
        overall_samples.add(sample)

        file_conditions[cond] += 1
        overall_conditions[cond] += 1

        overall_sample_conditions[sample][cond] += 1
        overall_sample_files[sample].add(file_name)

        rows_by_file_condition[(file_name, cond)] += 1
        rows_by_file_sample[(file_name, sample)] += 1

        if not pd.isna(cyc):
            cyc = int(cyc)
            overall_sample_cycles[sample].add(cyc)
            overall_cycle_minmax[sample][0] = min(overall_cycle_minmax[sample][0], cyc)
            overall_cycle_minmax[sample][1] = max(overall_cycle_minmax[sample][1], cyc)
            file_cycle_min = min(file_cycle_min, cyc)
            file_cycle_max = max(file_cycle_max, cyc)

        if idx_soc is not None:
            try:
                soc = float(row[idx_soc])
                if not np.isnan(soc):
                    overall_soc_minmax[sample][0] = min(overall_soc_minmax[sample][0], soc)
                    overall_soc_minmax[sample][1] = max(overall_soc_minmax[sample][1], soc)
            except:
                pass

    file_summaries.append({
        "file": file_name,
        "sheet_used": ws.title,
        "n_rows": file_rows,
        "n_columns": len(header),
        "columns": ", ".join([str(c) for c in header]),
        "n_unique_samples": len(file_samples),
        "min_sample": min(file_samples) if file_samples else None,
        "max_sample": max(file_samples) if file_samples else None,
        "n_unique_conditions": len(file_conditions),
        "cycle_min": None if file_cycle_min == np.inf else int(file_cycle_min),
        "cycle_max": None if file_cycle_max == -np.inf else int(file_cycle_max),
    })

    wb.close()


In [ ]:
# ------------------------------------------------------------
# 6) Build summary DataFrames
# ------------------------------------------------------------

file_summary_df = pd.DataFrame(file_summaries)

actual_samples = sorted(overall_samples)
actual_sample_set = set(actual_samples)

missing_vs_full = sorted(full_sample_set - actual_sample_set)
extra_vs_full = sorted(actual_sample_set - full_sample_set)

missing_vs_docx = sorted(docx_sample_set - actual_sample_set)
extra_vs_docx = sorted(actual_sample_set - docx_sample_set)

# Actual condition-level sample table
condition_rows = []
for cond, n_rows in overall_conditions.items():
    samples_in_cond = sorted([
        s for s, cond_counter in overall_sample_conditions.items()
        if cond in cond_counter
    ])

    condition_rows.append({
        "Observed_Label": condition_label(cond),
        "Observed_Charge_Rate": cond[0],
        "Observed_Temperature": cond[1],
        "Observed_Discharge_Rate": cond[2],
        "n_rows": n_rows,
        "n_unique_samples": len(samples_in_cond),
        "samples": ", ".join(map(str, samples_in_cond[:30])) + (" ..." if len(samples_in_cond) > 30 else "")
    })

actual_condition_df = pd.DataFrame(condition_rows).sort_values(
    ["Observed_Temperature", "Observed_Discharge_Rate", "Observed_Charge_Rate"],
    na_position="last"
)

# Sample-level table
sample_rows = []
for s in actual_samples:
    cond_counter = overall_sample_conditions[s]
    conds = list(cond_counter.keys())

    primary_cond = cond_counter.most_common(1)[0][0] if cond_counter else (None, None, None)
    primary_rows = cond_counter.most_common(1)[0][1] if cond_counter else 0

    full_exp = expected_full_map.get(s)
    docx_exp = expected_docx_map.get(s)

    cyc_set = overall_sample_cycles.get(s, set())
    cyc_min = min(cyc_set) if cyc_set else None
    cyc_max = max(cyc_set) if cyc_set else None
    cyc_count = len(cyc_set)

    soc_min, soc_max = overall_soc_minmax.get(s, [np.nan, np.nan])
    if soc_min == np.inf:
        soc_min = np.nan
    if soc_max == -np.inf:
        soc_max = np.nan

    observed_label = condition_label(primary_cond)

    if full_exp is not None:
        _, exp_cr, exp_temp, exp_rate, exp_label = full_exp
        match_full = (
            primary_cond[0] == exp_cr and
            primary_cond[1] == exp_temp and
            abs(float(primary_cond[2]) - float(exp_rate)) < 1e-6
        )
    else:
        exp_label = None
        match_full = False

    if docx_exp is not None:
        _, docx_cr, docx_temp, docx_rate, docx_label = docx_exp
        match_docx = (
            primary_cond[0] == docx_cr and
            primary_cond[1] == docx_temp and
            abs(float(primary_cond[2]) - float(docx_rate)) < 1e-6
        )
    else:
        docx_label = None
        match_docx = False

    sample_rows.append({
        "sample_number": s,
        "files_present": ", ".join(sorted(overall_sample_files[s])),
        "n_files_present": len(overall_sample_files[s]),
        "n_observed_conditions_for_sample": len(conds),
        "observed_primary_condition": observed_label,
        "observed_primary_rows": primary_rows,
        "expected_full_192_condition": exp_label,
        "expected_docx_condition": docx_label,
        "matches_full_192_map": match_full,
        "matches_docx_map": match_docx,
        "cycle_min": cyc_min,
        "cycle_max": cyc_max,
        "n_unique_cycles": cyc_count,
        "has_cycle_above_300": bool(cyc_max is not None and cyc_max > 300),
        "has_cycle_below_1": bool(cyc_min is not None and cyc_min < 1),
        "soc_min": soc_min,
        "soc_max": soc_max,
    })

sample_df = pd.DataFrame(sample_rows)


# Expected-vs-actual by condition, using DOCX condition table
docx_condition_summary = (
    expected_docx
    .groupby(["Condition", "Expected_Label", "Expected_Charge_Rate",
              "Expected_Temperature", "Expected_Discharge_Rate"])
    .agg(expected_docx_samples=("sample_number", lambda x: sorted(list(x))),
         n_expected_docx_samples=("sample_number", "nunique"))
    .reset_index()
)

full_condition_summary = (
    expected_full
    .groupby(["Condition", "Expected_Label", "Expected_Charge_Rate",
              "Expected_Temperature", "Expected_Discharge_Rate"])
    .agg(expected_full_samples=("sample_number", lambda x: sorted(list(x))),
         n_expected_full_samples=("sample_number", "nunique"))
    .reset_index()
)

def actual_samples_for_expected_condition(row):
    exp_cond = (
        row["Expected_Charge_Rate"],
        int(row["Expected_Temperature"]),
        round(float(row["Expected_Discharge_Rate"]), 3)
    )

    return sorted([
        s for s, counter in overall_sample_conditions.items()
        if exp_cond in counter
    ])

condition_compare = docx_condition_summary.merge(
    full_condition_summary[
        ["Condition", "n_expected_full_samples", "expected_full_samples"]
    ],
    on="Condition",
    how="left"
)

actual_samples_list = []
actual_n_list = []
for _, row in condition_compare.iterrows():
    a = actual_samples_for_expected_condition(row)
    actual_samples_list.append(a)
    actual_n_list.append(len(a))

condition_compare["actual_samples"] = actual_samples_list
condition_compare["n_actual_samples"] = actual_n_list

condition_compare["missing_vs_docx_condition"] = condition_compare.apply(
    lambda r: sorted(set(r["expected_docx_samples"]) - set(r["actual_samples"])),
    axis=1
)

condition_compare["extra_vs_docx_condition"] = condition_compare.apply(
    lambda r: sorted(set(r["actual_samples"]) - set(r["expected_docx_samples"])),
    axis=1
)

condition_compare["missing_vs_full_condition"] = condition_compare.apply(
    lambda r: sorted(set(r["expected_full_samples"]) - set(r["actual_samples"])),
    axis=1
)

condition_compare["actual_samples_str"] = condition_compare["actual_samples"].apply(lambda x: ", ".join(map(str, x)))
condition_compare["expected_docx_samples_str"] = condition_compare["expected_docx_samples"].apply(lambda x: ", ".join(map(str, x)))
condition_compare["expected_full_samples_str"] = condition_compare["expected_full_samples"].apply(lambda x: ", ".join(map(str, x)))
condition_compare["missing_vs_docx_condition_str"] = condition_compare["missing_vs_docx_condition"].apply(lambda x: ", ".join(map(str, x)))
condition_compare["extra_vs_docx_condition_str"] = condition_compare["extra_vs_docx_condition"].apply(lambda x: ", ".join(map(str, x)))
condition_compare["missing_vs_full_condition_str"] = condition_compare["missing_vs_full_condition"].apply(lambda x: ", ".join(map(str, x)))


# File overlap matrix
overlap_rows = []
file_to_samples = {}
for f in xlsx_files:
    file_to_samples[f] = set()
    for s, fileset in overall_sample_files.items():
        if f in fileset:
            file_to_samples[f].add(s)

for f1 in xlsx_files:
    for f2 in xlsx_files:
        overlap = sorted(file_to_samples[f1].intersection(file_to_samples[f2]))
        overlap_rows.append({
            "file_1": f1,
            "file_2": f2,
            "n_overlap_samples": len(overlap),
            "overlap_samples": ", ".join(map(str, overlap[:50])) + (" ..." if len(overlap) > 50 else "")
        })

overlap_df = pd.DataFrame(overlap_rows)


# Problem flags
problem_rows = []

# Sample appears in more than one observed condition
multi_condition_samples = sample_df[sample_df["n_observed_conditions_for_sample"] > 1]
for _, r in multi_condition_samples.iterrows():
    problem_rows.append({
        "issue_type": "Sample appears under multiple observed conditions",
        "sample_number": r["sample_number"],
        "detail": r["observed_primary_condition"]
    })

# Sample-condition mismatch
mismatch_full = sample_df[sample_df["matches_full_192_map"] == False]
for _, r in mismatch_full.iterrows():
    problem_rows.append({
        "issue_type": "Sample does not match full 192 expected condition map",
        "sample_number": r["sample_number"],
        "detail": f"Observed={r['observed_primary_condition']} | Expected={r['expected_full_192_condition']}"
    })

# Cycle above 300
above_300 = sample_df[sample_df["has_cycle_above_300"] == True]
for _, r in above_300.iterrows():
    problem_rows.append({
        "issue_type": "Cycle_Index above 300",
        "sample_number": r["sample_number"],
        "detail": f"cycle_min={r['cycle_min']}, cycle_max={r['cycle_max']}, n_unique_cycles={r['n_unique_cycles']}"
    })

problem_df = pd.DataFrame(problem_rows)


In [ ]:
# ------------------------------------------------------------
# 7) Automatic conclusion
# ------------------------------------------------------------

n_samples = len(actual_sample_set)
n_conditions = len(overall_conditions)
max_cycle = sample_df["cycle_max"].max() if not sample_df.empty else None
min_cycle = sample_df["cycle_min"].min() if not sample_df.empty else None

matches_full_sample_set = (actual_sample_set == full_sample_set)
matches_docx_sample_set = (actual_sample_set == docx_sample_set)

all_samples_match_full_conditions = bool(sample_df["matches_full_192_map"].all()) if not sample_df.empty else False
all_samples_match_docx_conditions = bool(sample_df["matches_docx_map"].all()) if not sample_df.empty else False

if matches_full_sample_set and all_samples_match_full_conditions:
    dataset_status = "FULL_192_MATCH"
    dataset_interpretation = (
        "Veri seti tam 192 hücrelik teorik tasarımla uyumlu görünüyor."
    )
elif matches_docx_sample_set and all_samples_match_docx_conditions:
    dataset_status = "DOCX_REDUCED_182_MATCH"
    dataset_interpretation = (
        "Veri seti DOCX'teki azaltılmış haritayla uyumlu görünüyor: "
        "ilk 22 koşulda 8'er hücre, Condition 23'te 177-179, "
        "Condition 24'te 185-187. Yani fiilen 182 hücre var."
    )
else:
    dataset_status = "CUSTOM_OR_INCONSISTENT_SUBSET"
    dataset_interpretation = (
        "Veri seti ne tam 192 hücrelik tasarımla ne de DOCX'teki azaltılmış "
        "182 hücrelik haritayla birebir örtüşüyor. Eksik/fazla sample listelerine bakılmalı."
    )

if max_cycle is not None and max_cycle <= 300:
    cycle_interpretation = (
        f"Cycle_Index maksimum {int(max_cycle)}. Bu, dosyaların ilk 300 çevrimle "
        "sınırlandırılmış olduğunu destekler."
    )
else:
    cycle_interpretation = (
        f"Cycle_Index maksimum {max_cycle}. Bu, dosyalarda 300 çevrimden sonraki "
        "kayıtların da bulunduğunu gösterir."
    )

conclusion_df = pd.DataFrame([
    {"Metric": "Total uploaded Excel files", "Value": len(xlsx_files)},
    {"Metric": "Total rows across files", "Value": int(file_summary_df["n_rows"].sum())},
    {"Metric": "Unique sample_number count", "Value": n_samples},
    {"Metric": "Observed unique condition count", "Value": n_conditions},
    {"Metric": "Min sample_number", "Value": min(actual_samples) if actual_samples else None},
    {"Metric": "Max sample_number", "Value": max(actual_samples) if actual_samples else None},
    {"Metric": "Matches full 192 sample set", "Value": matches_full_sample_set},
    {"Metric": "Matches DOCX reduced sample set", "Value": matches_docx_sample_set},
    {"Metric": "Missing samples vs full 192", "Value": ", ".join(map(str, missing_vs_full))},
    {"Metric": "Extra samples vs full 192", "Value": ", ".join(map(str, extra_vs_full))},
    {"Metric": "Missing samples vs DOCX reduced", "Value": ", ".join(map(str, missing_vs_docx))},
    {"Metric": "Extra samples vs DOCX reduced", "Value": ", ".join(map(str, extra_vs_docx))},
    {"Metric": "Min Cycle_Index", "Value": min_cycle},
    {"Metric": "Max Cycle_Index", "Value": max_cycle},
    {"Metric": "Dataset status", "Value": dataset_status},
    {"Metric": "Dataset interpretation", "Value": dataset_interpretation},
    {"Metric": "Cycle interpretation", "Value": cycle_interpretation},
])

In [ ]:
# ------------------------------------------------------------
# 8) Print concise result
# ------------------------------------------------------------

print("\n" + "="*80)
print("FINAL DATASET CHECK")
print("="*80)

print(f"Toplam satır sayısı: {int(file_summary_df['n_rows'].sum()):,}")
print(f"Farklı sample_number sayısı: {n_samples}")
print(f"Gözlenen farklı koşul sayısı: {n_conditions}")
print(f"Sample aralığı: {min(actual_samples) if actual_samples else None} - {max(actual_samples) if actual_samples else None}")
print(f"Cycle_Index aralığı: {min_cycle} - {max_cycle}")

print("\nDurum:")
print(dataset_status)
print(dataset_interpretation)
print(cycle_interpretation)

print("\nFull 192 tasarıma göre eksik sample'lar:")
print(missing_vs_full)

print("\nDOCX azaltılmış 182 haritaya göre eksik sample'lar:")
print(missing_vs_docx)

print("\nDOCX azaltılmış 182 haritaya göre fazla sample'lar:")
print(extra_vs_docx)

print("\nKoşul bazında gerçek sample sayıları:")
display(condition_compare[[
    "Condition",
    "Expected_Label",
    "n_expected_full_samples",
    "n_expected_docx_samples",
    "n_actual_samples",
    "actual_samples_str",
    "missing_vs_docx_condition_str",
    "extra_vs_docx_condition_str",
    "missing_vs_full_condition_str"
]])

print("\nDosya bazında özet:")
display(file_summary_df)

print("\nSample bazında ilk 20 satır:")
display(sample_df.head(20))

if problem_df.empty:
    print("\nProblem flag bulunmadı.")
else:
    print("\nProblem flag bulundu:")
    display(problem_df.head(100))

In [ ]:
# ============================================================
# Save detailed Excel report
# ============================================================

report_name = os.path.join(
    OUTPUT_DIR,
    "CALCE_Processed_Outputv3_dataset_check_report.xlsx"
)

with pd.ExcelWriter(report_name, engine="openpyxl") as writer:
    conclusion_df.to_excel(writer, index=False, sheet_name="00_Conclusion")
    file_summary_df.to_excel(writer, index=False, sheet_name="01_File_Summary")
    condition_compare.to_excel(writer, index=False, sheet_name="02_Condition_Check")
    sample_df.to_excel(writer, index=False, sheet_name="03_Sample_Check")
    actual_condition_df.to_excel(writer, index=False, sheet_name="04_Observed_Conditions")
    overlap_df.to_excel(writer, index=False, sheet_name="05_File_Overlap")
    expected_full.to_excel(writer, index=False, sheet_name="06_Expected_Full_192")
    expected_docx.to_excel(writer, index=False, sheet_name="07_Expected_DOCX_182")
    problem_df.to_excel(writer, index=False, sheet_name="08_Problem_Flags")

print(f"\nReport saved: {report_name}")

# Optional Colab download
if IN_COLAB and files is not None:
    try:
        files.download(report_name)
    except Exception:
        pass

In [ ]:
## Expected manuscript-level checks

This notebook is expected to confirm that the processed working files are a custom processed subset rather than the full nominal 192-cell design. The main manuscript-level checks are:

- processed records: 2,358,728
- unique sample numbers: 181
- sample range: 1--187
- cycle-index range: 1--611
- missing samples relative to the reduced condition map: 24 and 31
- extra sample relative to the reduced condition map: 180

The detailed Excel report is written to:

`outputs/01_dataset_consistency_check/CALCE_Processed_Outputv3_dataset_check_report.xlsx`